In [ ]:
import pandas as pd
import json
from confluent_kafka import Producer
from time import time

# Configuration
KAFKA_BOOTSTRAP_SERVERS = 'localhost:9092'
TOPIC = 'green-trips'
FILE_PATH = 'green_tripdata_2025-10.parquet'

# Load data
df = pd.read_parquet(FILE_PATH)

# Select specific columns
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]
df = df[columns]

# Handle NaN values (fill with 0 for numerics, handle datetimes)
df['passenger_count'] = df['passenger_count'].fillna(0)
# Note: Datetimes in pandas read_parquet are usually datetime objects. 
# We need to convert them to strings for JSON serialization.

def delivery_report(err, msg):
    """ Called once for each message produced to indicate delivery result. """
    if err is not None:
        print(f'Message delivery failed: {err}')
    # else:
    #     print(f'Message delivered to {msg.topic()} [{msg.partition()}]')

# Create Producer
p = Producer({'bootstrap.servers': KAFKA_BOOTSTRAP_SERVERS})

t0 = time()

# Send rows
for index, row in df.iterrows():
    # Convert row to dictionary
    record = row.to_dict()
    
    # Convert timestamps to string (ISO format or specific format)
    # Flink expects 'yyyy-MM-dd HH:mm:ss'
    record['lpep_pickup_datetime'] = record['lpep_pickup_datetime'].strftime('%Y-%m-%d %H:%M:%S')
    record['lpep_dropoff_datetime'] = record['lpep_dropoff_datetime'].strftime('%Y-%m-%d %H:%M:%S')
    
    # Serialize to JSON
    value = json.dumps(record)
    
    # Send
    p.poll(0) # Callback trigger
    p.produce(TOPIC, value.encode('utf-8'), callback=delivery_report)

# Flush
p.flush()
t1 = time()

print(f'took {(t1 - t0):.2f} seconds')